# INT8 Post-Training_quantization (PTQ)
**Project:** Evaluating Neural Network Quantization for Efficient Deep Learning Inference

**Goal:** Quantize a pretrained ResNet-18 model to INT8 without additional training, then compare its predictive performance, serialized model size, and CPU inference latency against the FP32 baseline.

**Measurements:** Test accuracy, serialized model size, and CPU inference latency.

**Calculations:** Accuracy change, size reduction, and latency improvement, all compared to FP32 baseline.

**Dataset:** CIFAR-10

**Model:** Pretrained ResNet-18 fine-tuned for CIFAR-10 (SamAdamDay/resnet18_cifar10).

**Evaluation size:** The same fixed 1000 test images used in the FP32 baseline.


# Configuring the Colab notebook
Note: Select **CPU** for runtime if available

In [ ]:
import platform
import torch
import torchvision

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU") # Use CPU if available

# Installing the module-loading library

In [ ]:
%pip install -q timm # Loading pretrained image-classification model from Hugging Face

import copy
import os
import random
import time
import json
import statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Setup complete.")

# Checking TorchAO compability for quantization

In [ ]:
%pip install -q torchao
import torch
import torchao

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)

from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e,
)

from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import (
    X86InductorQuantizer,
    get_default_x86_inductor_quantization_config,
)

print("PT2E quantization imports successful.")

# Loading CIFAR-10 dataset for the ResNet-18
Note the cell below might take approx. 25 minutes when first ran

In [ ]:
# Loading CIFAR-10 dataset for the ResNet-18 model
from torch.utils.data import DataLoader, Subset

DATA_DIR = "./data"
BATCH_SIZE = 32
EVAL_SIZE = 1000

# Using the preprocessing specified by the model card
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.49139968, 0.48215827, 0.44653124),
        std=(0.24703233, 0.24348505, 0.26158768)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

# Fixed and reproducible subset of the official test split
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(
    len(test_dataset), generator=generator
)[:EVAL_SIZE].tolist()

test_subset = Subset(test_dataset, indices)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Training images available:", len(train_dataset))
print("Official test images:", len(test_dataset))
print("Images in this evaluation:", len(test_subset))


# Loading the pre-trained ResNet-18 model

In [ ]:
MODEL_ID = "hf_hub:SamAdamDay/resnet18_cifar10"

model = timm.create_model(
    MODEL_ID,
    pretrained=True
)

model.eval()

print("Model loaded successfully.")
print("Architecture:", model.__class__.__name__)
print("Number of parameters:",
      sum(p.numel() for p in model.parameters()))
print("Number of output classes:", model.num_classes)